## Import

In [4]:
import re
import os 
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from pyhive import presto
from datetime import date
from collections import Counter

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'

conn=presto.connect('presto-gateway.processing.data.production.internal',presto_port,username)

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
connection4= presto.connect('processing.processing.data.production.internal',presto_port,username)


pd.set_option('display.max_columns', None)

## Overall

In [6]:
overall_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916' 
        and event_props_campaign_id = '1725970812' 
),

merge as(
        select rr_net.*,view_tbl.event_type,click_tbl.click_type
        from rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        )

SELECT
        test_group,
        -- event_type,
        -- click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1
        )
    ORDER BY 1
    
    
"""
overall=pd.read_sql(overall_query,conn)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62370/2770846961.py:154: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall=pd.read_sql(overall_query,conn)


In [7]:
overall

,test_group,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,426083,1495502,726604,48.59,66.94,106,132,310,174,14,1.63,20.10,17.98,11.31,24367,300574,268872,169149
1,test,426987,1499498,726049,48.42,66.75,106,131,311,174,14,1.63,20.14,17.95,11.46,24489,302038,269165,171869


In [11]:
overall.to_clipboard(index=False)

## Level 1 breakdown

In [9]:
overall_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916' 
        and event_props_campaign_id = '1725970812' 
),

merge as(
        select rr_net.*,view_tbl.event_type,click_tbl.click_type
        from rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        )

SELECT
        test_group,
        event_type,
        click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3

    
    
    
    
"""
overall1=pd.read_sql(overall_query,conn)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62370/3292753032.py:157: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall1=pd.read_sql(overall_query,conn)


In [12]:
overall1

,test_group,event_type,click_type,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,426083,1495502,726604,48.59,66.94,106,132,310,174,14,1.63,20.10,17.98,11.31,24367,300574,268872,169149
1,test,viewed,close_click,315748,826810,400370,48.42,66.68,112,134,319,178,14,1.58,20.10,17.88,11.63,13062,166171,147818,96118
2,test,viewed,play_click,32360,68335,30131,44.09,63.98,104,130,319,172,15,1.94,20.71,19.46,13.34,1327,14152,13301,9118
3,test,viewed,None,10982,22090,11297,51.14,72.39,110,132,315,191,12,1.16,15.00,20.79,11.41,257,3314,4593,2520
4,test,None,None,200427,582671,284380,48.81,66.94,98,128,300,167,14,1.69,20.34,17.77,11.02,9851,118510,103539,64186


In [14]:
overall1.to_clipboard(index=False)

## Pre-Post

In [15]:
overall_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240905' AND yyyymmdd <= '20240910'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240905' AND yyyymmdd <= '20240910' 
        and event_props_campaign_id = '1725970812' 
),

merge as(
        select rr_net.*,view_tbl.event_type,click_tbl.click_type
        from rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        )

SELECT
        test_group,
        -- event_type,
        -- click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1
        )
    ORDER BY 1
    
    
"""
pre_overall=pd.read_sql(overall_query,conn)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62370/2275741643.py:154: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pre_overall=pd.read_sql(overall_query,conn)


In [16]:
pre_overall

,test_group,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,378291,1291512,695940,53.89,73.18,104,128,299,177,13,1.16,16.99,18.89,8.66,14974,219455,243923,111799
1,test,377944,1289573,695496,53.93,73.13,104,128,302,178,14,1.10,17.01,18.81,8.74,14159,219396,242615,112735


In [17]:
pre_overall.to_clipboard(index=False)

## Cobra distr

In [27]:
overall_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916' 
        and event_props_campaign_id = '1725970812' 
),

merge as(
        select rr_net.*,view_tbl.event_type,click_tbl.click_type
        from rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        )

SELECT
        test_group,
        -- event_type,
        -- click_type,
        cobra_ttc_bucket,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            -- event_type,
            -- click_type,


            CASE 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec <= 15 THEN 'a. 0-15 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 16 AND 30 THEN 'b. 16-30 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 31 AND 60 THEN 'c. 31-60 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec BETWEEN 61 AND 180 THEN 'd. 61-180 sec' 
            WHEN modified_order_status = 'COBRA' AND time_spent_in_sec > 180 THEN 'e. 180+ sec' 
            WHEN modified_order_status != 'COBRA' THEN 'f. NO COBRA'
            ELSE 'Check' END cobra_ttc_bucket,
            
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)

           and click_type = 'play_click'

           
        GROUP BY 1,2
        )
    ORDER BY 1,2
    
    
"""
overall_ttc_breakup=pd.read_sql(overall_query,conn)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62370/543812215.py:170: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_ttc_breakup=pd.read_sql(overall_query,conn)


In [28]:
overall_ttc_breakup

,test_group,cobra_ttc_bucket,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,test,a. 0-15 sec,650,701,0,0.00,0.00,NaN,11.0,NaN,11,NaN,0.00,100.0,0.00,0.00,0,701,0,0
1,test,b. 16-30 sec,774,823,0,0.00,0.00,NaN,22.0,NaN,22,NaN,0.00,100.0,0.00,0.00,0,823,0,0
2,test,c. 31-60 sec,1299,1527,0,0.00,0.00,NaN,46.0,NaN,46,NaN,0.00,100.0,0.00,0.00,0,1527,0,0
3,test,d. 61-180 sec,4432,6538,0,0.00,0.00,NaN,116.0,NaN,116,NaN,0.00,100.0,0.00,0.00,0,6538,0,0
4,test,e. 180+ sec,3577,4563,0,0.00,0.00,NaN,251.0,NaN,251,NaN,0.00,100.0,0.00,0.00,0,4563,0,0
5,test,f. NO COBRA,30317,54183,30131,55.61,80.69,104.0,NaN,318.0,281,15.0,2.45,0.0,24.55,16.83,1327,0,13301,9118


In [29]:
overall_ttc_breakup.gross_count.sum()

68335

In [30]:
68335-54183

14152

In [31]:
overall_ttc_breakup.to_clipboard(index=False)

## Other

In [145]:
overall_daywise_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916' 
        and event_props_campaign_id = '1725970812' 
),

merge as(
        select rr_net.*,view_tbl.event_type,click_tbl.click_type
        from rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        )

SELECT
        yyyymmdd,
        test_group,
        event_type,
        click_type,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            yyyymmdd,
            test_group,
            event_type,
            click_type,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

    
    
    
    
"""
overall_daywise_df=pd.read_sql(overall_daywise_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/3833155388.py:159: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_daywise_df=pd.read_sql(overall_daywise_query,conn)


In [146]:
overall_daywise_df

,yyyymmdd,test_group,event_type,click_type,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,20240911,control,None,None,132673,248221,135276,54.50,73.96,100,124,297,174,14,1.18,16.82,19.04,8.02,2934,41763,47257,19906
1,20240911,test,viewed,close_click,76793,140182,76298,54.43,73.71,115,127,308,177,13,1.20,17.10,18.90,7.98,1686,23968,26499,11181
2,20240911,test,viewed,play_click,6688,12582,6363,50.57,71.67,98,116,289,166,15,1.24,17.95,20.73,9.14,156,2258,2608,1150
3,20240911,test,viewed,None,2323,4137,2446,59.12,79.79,116,128,297,192,12,0.70,11.48,20.23,7.98,29,475,837,330
4,20240911,test,None,None,46714,91191,49892,54.71,74.19,92,121,290,170,15,1.28,16.85,19.06,7.66,1165,15368,17378,6989
5,20240912,control,None,None,131658,263152,122118,46.41,65.24,103,127,313,169,17,1.60,22.36,18.46,10.78,4202,58828,48589,28369
6,20240912,test,viewed,close_click,66587,132937,60498,45.51,64.54,109,129,322,173,17,1.57,22.53,18.64,11.35,2082,29950,24774,15083
7,20240912,test,viewed,play_click,5880,12103,5069,41.88,62.51,104,126,315,167,18,1.85,22.75,20.17,12.85,224,2753,2441,1555
8,20240912,test,viewed,None,1872,3681,1721,46.75,68.57,119,131,323,188,15,1.28,18.28,21.49,11.84,47,673,791,436
9,20240912,test,None,None,57945,117056,55593,47.49,65.79,98,125,294,163,17,1.54,22.55,17.94,10.11,1807,26391,21001,11833


In [150]:
overall_daywise_df.to_clipboard(index=False)

In [132]:
overall_daywise_df.to_csv('overall_daywise_df.csv',index=False)

In [151]:
session_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916' 
        and event_props_campaign_id = '1725970812' 
),

merge as(
        select rr_net.*,view_tbl.event_type,click_tbl.click_type
        from rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        )

SELECT
        test_group,
        event_type,
        click_type,
        time_period,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

    
    
    
    
"""
session_df=pd.read_sql(session_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/3233821542.py:165: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  session_df=pd.read_sql(session_query,conn)


In [152]:
session_df

,test_group,event_type,click_type,time_period,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,afternoon,205733,438073,250827,57.26,76.61,105,127,309,183,11,1.25,14.22,18.96,7.91,5494,62300,83060,34667
1,control,None,None,evening,193280,478646,187429,39.16,56.51,103,132,322,169,16,1.53,25.31,17.01,16.63,7301,121148,81417,79600
2,control,None,None,morning,133869,299946,158523,52.85,69.56,107,132,286,168,15,1.10,20.12,16.42,9.22,3295,60354,49255,27641
3,control,None,None,rest_evening,74195,137305,64916,47.28,69.02,97,128,323,180,16,1.84,19.88,21.10,9.13,2528,27293,28978,12531
4,control,None,None,rest_morning,69515,141250,64773,45.86,64.72,113,139,308,173,12,4.07,20.82,18.49,10.40,5742,29409,26113,14688
5,test,viewed,close_click,afternoon,139382,252121,143576,56.95,76.14,115,130,318,187,11,1.15,14.48,18.80,8.23,2906,36504,47396,20737
6,test,viewed,close_click,evening,130781,266185,103082,38.73,56.05,109,134,328,172,15,1.54,25.31,16.98,17.08,4104,67368,45211,45453
7,test,viewed,close_click,morning,89817,158865,83933,52.83,69.48,113,134,292,170,15,1.06,20.08,16.35,9.38,1677,31903,25980,14909
8,test,viewed,close_click,rest_evening,45878,74289,34999,47.11,68.55,97,130,331,183,16,1.90,19.85,20.80,9.57,1408,14745,15452,7106
9,test,viewed,close_click,rest_morning,42764,75255,34727,46.15,64.76,118,141,326,178,12,3.94,20.79,18.27,10.51,2965,15643,13750,7910


In [153]:
session_df.to_clipboard(index=False)

In [135]:
session_df.to_csv('session_df.csv',index=False)

In [136]:
ltr_segment_games_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240915' 
        and event_props_campaign_id = '1725970812' 
),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-11'
    
),

merge as(
        select rr_net.*,
        view_tbl.event_type,
        click_tbl.click_type,
        segment.ltr_segment
        from 
            rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        LEFT JOIN 
            segment
        ON rr_net.customer_id=segment.customer_id
        )

SELECT
        test_group,
        event_type,
        click_type,
        ltr_segment,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            ltr_segment,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

    
    
    
    
"""
ltr_segment_games_df=pd.read_sql(ltr_segment_games_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/3587971687.py:182: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ltr_segment_games_df=pd.read_sql(ltr_segment_games_query,conn)


In [137]:
ltr_segment_games_df

,test_group,event_type,click_type,ltr_segment,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,HH,64553,149254,63920,42.83,62.85,114,144,361,190,15,2.23,20.46,19.54,14.43,3335,30531,29163,21530
1,control,None,None,PHH,285313,1050787,499143,47.50,65.02,105,130,302,170,14,1.44,21.81,17.19,11.71,15154,229144,180613,123061
2,control,None,None,UNKNOWN,9526,24583,5458,22.20,55.38,86,133,304,168,14,7.75,19.54,32.59,17.30,1906,4803,8011,4252
3,control,None,None,None,20818,45829,18648,40.69,62.22,116,147,437,213,14,2.71,18.73,20.87,16.30,1243,8584,9565,7469
4,test,viewed,close_click,HH,48780,97137,41451,42.67,62.48,127,145,370,195,14,2.17,20.10,19.35,15.21,2105,19526,18793,14776
5,test,viewed,close_click,PHH,214492,560687,264166,47.11,64.52,111,132,311,172,15,1.43,21.99,17.07,12.04,8035,123306,95736,67506
6,test,viewed,close_click,UNKNOWN,5578,12573,3028,24.08,55.58,90,139,269,162,13,5.78,20.09,30.85,18.50,727,2526,3879,2326
7,test,viewed,close_click,None,11718,25010,9913,39.64,60.94,111,150,414,210,14,2.85,19.36,20.68,16.82,713,4841,5173,4207
8,test,viewed,play_click,HH,4707,8819,3482,39.48,61.75,138,143,392,203,15,2.36,19.65,21.65,16.20,208,1733,1909,1429
9,test,viewed,play_click,PHH,21236,46149,20212,43.80,62.10,104,127,300,164,16,1.61,22.64,17.96,13.62,743,10447,8290,6287


In [138]:
retention_segment_games_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240915' 
        and event_props_campaign_id = '1725970812' 
),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-11'
    
),

merge as(
        select rr_net.*,
        view_tbl.event_type,
        click_tbl.click_type,
        segment.retention_segment
        from 
            rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        LEFT JOIN 
            segment
        ON rr_net.customer_id=segment.customer_id
        )

SELECT
        test_group,
        event_type,
        click_type,
        retention_segment,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            retention_segment,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

    
    
    
    
"""
retention_segment_games_df=pd.read_sql(retention_segment_games_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/3682328015.py:182: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  retention_segment_games_df=pd.read_sql(retention_segment_games_query,conn)


In [139]:
retention_segment_games_df

,test_group,event_type,click_type,retention_segment,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,DORMANT,53528,112442,44427,39.51,60.25,117.0,141,343,186,15,2.07,22.03,20.27,15.61,2324,24776,22796,17556
1,control,None,None,ELITE,61979,387715,194261,50.10,66.84,97.0,124,279,160,15,1.20,21.62,16.44,10.33,4636,83819,63722,40055
2,control,None,None,GOLD,64497,189977,88117,46.38,64.21,110.0,133,322,174,14,1.64,21.92,17.48,12.20,3125,41648,33216,23173
3,control,None,None,HH,36905,93157,43364,46.55,64.91,114.0,142,357,187,14,2.05,20.00,17.94,13.01,1911,18631,16711,12123
4,control,None,None,INACTIVE,25267,50782,19305,38.02,60.09,110.0,150,366,197,15,2.37,20.91,21.56,16.57,1206,10621,10948,8417
5,control,None,None,PLATINUM,68828,259935,121979,46.93,64.32,102.0,128,295,165,14,1.43,22.55,17.09,11.68,3729,58625,44414,30348
6,control,None,None,PRIME,3418,23178,12582,54.28,70.48,97.0,121,266,156,14,1.08,20.77,15.88,7.65,250,4815,3681,1774
7,control,None,None,SILVER,35444,82855,39028,47.10,64.74,125.0,141,337,187,14,1.58,20.20,17.24,13.45,1308,16740,14288,11145
8,control,None,None,UNKNOWN,9526,24583,5458,22.20,55.38,86.0,133,305,168,14,7.75,19.54,32.59,17.30,1906,4803,8011,4252
9,control,None,None,None,20818,45829,18648,40.69,62.22,116.0,147,437,213,14,2.71,18.73,20.87,16.30,1243,8584,9565,7469


In [140]:
service_games_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240915'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240915' 
        and event_props_campaign_id = '1725970812' 
),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-11'
    
),

merge as(
        select rr_net.*,
        view_tbl.event_type,
        click_tbl.click_type,
        segment.retention_segment
        from 
            rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        LEFT JOIN 
            segment
        ON rr_net.customer_id=segment.customer_id
        )

SELECT
        test_group,
        event_type,
        click_type,
        service_category,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            service_category,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

        
           
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4

    
    
    
    
"""
service_games_df=pd.read_sql(service_games_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/1397612730.py:188: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  service_games_df=pd.read_sql(service_games_query,conn)


In [141]:
service_games_df

,test_group,event_type,click_type,service_category,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,None,None,auto,159326,405395,136774,33.74,47.64,112,129,289,153,15,2.91,27.80,13.86,21.64,11789,112712,56197,87719
1,control,None,None,bike,235395,625683,353743,56.54,76.42,104,128,314,184,15,1.13,17.04,19.22,5.39,7056,106629,120262,33714
2,control,None,None,cab,106732,239375,96652,40.38,61.77,81,147,340,196,13,1.17,22.44,21.26,14.57,2793,53721,50893,34879
3,test,viewed,close_click,auto,110778,230277,77616,33.71,47.52,116,131,297,155,15,2.98,27.56,13.77,21.94,6862,63453,31707,50519
4,test,viewed,close_click,bike,164673,327911,185867,56.68,76.62,112,130,326,188,15,1.03,16.85,19.26,5.47,3374,55258,63154,17948
5,test,viewed,close_click,cab,71323,137219,55075,40.14,61.19,92,151,354,203,14,0.98,22.95,20.93,14.83,1344,31488,28720,20348
6,test,viewed,play_click,auto,9908,18947,5713,30.15,44.29,111,129,295,151,15,3.44,27.82,14.07,24.44,651,5271,2665,4631
7,test,viewed,play_click,bike,16615,28989,15159,52.29,74.17,97,123,324,182,16,1.43,17.96,21.13,6.42,415,5205,6126,1861
8,test,viewed,play_click,cab,6225,11315,4049,35.78,58.91,58,149,337,200,14,1.18,22.19,23.04,17.66,133,2511,2607,1998
9,test,viewed,None,auto,3084,5616,2118,37.71,53.85,121,134,294,164,13,1.98,21.37,16.04,22.79,111,1200,901,1280


In [179]:
overall_cluster_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (              
                'Gurgaon Rural', 
                'NCR N sec 6354', 
                'Dashrathpuri_Del_East', 
                'NCR N sec 2262', 
                'NCR N Sharda University 2', 
                'NCR N Gharoli', 
                'NCR G Vijay Nagar 2', 
                'NCR N Venice Mall', 
                'Dwarka_Del', 
                'NCR N WaveCity Center'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916' 
        and event_props_campaign_id = '1725970812' 
),

merge as(
        select rr_net.*,view_tbl.event_type,click_tbl.click_type
        from rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        )

SELECT
        test_group,
        event_type,
        click_type,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            cluster_name,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

                    
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4   
    
"""
overall_cluster=pd.read_sql(overall_cluster_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/4261486077.py:168: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_cluster=pd.read_sql(overall_cluster_query,conn)


In [182]:
overall_cluster

,test_group,event_type,click_type,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
18,test,viewed,play_click,Dashrathpuri_Del_East,696,1195,530,44.35,66.03,178.0,143,342,194,19,0.42,24.02,21.34,9.54,5,287,255,114
19,test,viewed,play_click,Dwarka_Del,213,315,159,50.48,67.94,201.0,124,204,145,12,0.63,17.46,16.19,13.97,2,55,51,44
20,test,viewed,play_click,Gurgaon Rural,316,554,238,42.96,60.65,37.0,141,317,176,14,0.18,25.81,17.51,13.36,1,143,97,74
21,test,viewed,play_click,NCR G Vijay Nagar 2,474,640,408,63.75,81.09,NaN,126,283,176,12,0.00,15.00,17.19,3.91,0,96,110,25
22,test,viewed,play_click,NCR N Gharoli,673,972,540,55.56,77.26,76.0,135,367,214,15,0.62,14.09,21.09,8.02,6,137,205,78
23,test,viewed,play_click,NCR N Sharda University 2,536,1054,299,28.37,43.64,58.0,107,296,123,10,1.23,40.80,14.99,14.33,13,430,158,151
24,test,viewed,play_click,NCR N Venice Mall,377,866,280,32.33,46.88,145.0,121,255,143,10,7.27,33.60,13.97,12.24,63,291,121,106
25,test,viewed,play_click,NCR N WaveCity Center,273,394,216,54.82,78.43,188.0,121,377,211,12,0.25,15.48,22.84,5.84,1,61,90,23
26,test,viewed,play_click,Other_clusters,30158,62339,27458,44.05,64.05,103.0,130,318,174,15,1.98,20.30,19.59,13.64,1235,12652,12214,8502


In [189]:
overall_cluster=overall_cluster[overall_cluster['click_type']=='play_click']

In [190]:
overall_cluster.to_clipboard(index=False)

In [183]:
overall_cluster_session_query=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (
                'Gurgaon Rural', 
                'NCR N sec 6354', 
                'Dashrathpuri_Del_East', 
                'NCR N sec 2262', 
                'NCR N Sharda University 2', 
                'NCR N Gharoli', 
                'NCR G Vijay Nagar 2', 
                'NCR N Venice Mall', 
                'Dwarka_Del', 
                'NCR N WaveCity Center'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
),





view_tbl as(
        select 
            profile_identity as user_id,
            yyyymmdd,
            'viewed' as event_type
        from
            clevertap.customer_notification_viewed_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        and event_props_campaign_id = '1725970812' 

),

click_tbl as(
        select
            profile_identity as user_id,
            CASE
                when event_props_wzrk_c2a='Button 2' then 'play_click'
                else 'close_click'
            end as click_type,
            yyyymmdd
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916' 
        and event_props_campaign_id = '1725970812' 
),

merge as(
        select rr_net.*,view_tbl.event_type,click_tbl.click_type
        from rr_net
        left join 
            view_tbl
                on rr_net.customer_id=view_tbl.user_id
                and rr_net.yyyymmdd=view_tbl.yyyymmdd
        left join 
            click_tbl
            on view_tbl.user_id=click_tbl.user_id
            and view_tbl.yyyymmdd=click_tbl.yyyymmdd
        )

SELECT
        test_group,
        event_type,
        click_type,
        time_period,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            test_group,
            event_type,
            click_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            cluster_name,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

                    
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
           and NOT (test_group = 'control' AND event_type IS NOT NULL)
           and NOT (event_type IS NULL AND click_type IS NOT NULL)
        GROUP BY 1,2,3,4,5
        )
    ORDER BY 1,2,3,4,5  
    
"""
overall_cluster_session=pd.read_sql(overall_cluster_session_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/1746420347.py:176: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overall_cluster_session=pd.read_sql(overall_cluster_session_query,conn)


In [184]:
overall_cluster_session=overall_cluster_session[overall_cluster_session['click_type']=='play_click']

In [185]:
overall_cluster_session

,test_group,event_type,click_type,time_period,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
90,test,viewed,play_click,afternoon,Dashrathpuri_Del_East,262,440,192,43.64,67.95,249.0,148.0,324.0,212.0,19,0.23,23.41,24.09,8.41,1,103,106,37
91,test,viewed,play_click,afternoon,Dwarka_Del,79,99,62,62.63,75.76,NaN,147.0,556.0,165.0,8,0.00,12.12,12.12,12.12,0,12,12,12
92,test,viewed,play_click,afternoon,Gurgaon Rural,134,221,93,42.08,61.54,NaN,137.0,394.0,191.0,14,0.00,24.89,19.46,13.57,0,55,43,30
93,test,viewed,play_click,afternoon,NCR G Vijay Nagar 2,170,196,162,82.65,94.39,NaN,125.0,267.0,173.0,11,0.00,5.10,11.73,0.51,0,10,23,1
94,test,viewed,play_click,afternoon,NCR N Gharoli,220,269,182,67.66,91.82,198.0,88.0,384.0,296.0,10,0.74,4.46,23.79,2.97,2,12,64,8
95,test,viewed,play_click,afternoon,NCR N Sharda University 2,287,498,154,30.92,47.39,31.0,103.0,303.0,116.0,9,0.20,40.16,16.47,12.25,1,200,82,61
96,test,viewed,play_click,afternoon,NCR N Venice Mall,153,263,117,44.49,65.40,150.0,119.0,227.0,154.0,8,6.46,17.11,20.15,11.03,17,45,53,29
97,test,viewed,play_click,afternoon,NCR N WaveCity Center,77,97,67,69.07,95.88,NaN,117.0,464.0,426.0,9,0.00,3.09,24.74,1.03,0,3,24,1
98,test,viewed,play_click,afternoon,Other_clusters,11680,18762,9941,52.98,74.52,96.0,126.0,319.0,186.0,12,1.55,14.39,21.13,9.54,291,2699,3964,1789
99,test,viewed,play_click,evening,Dashrathpuri_Del_East,192,309,122,39.48,58.90,180.0,153.0,435.0,182.0,17,0.32,26.86,19.09,13.92,1,83,59,43


In [193]:
overall_cluster_session.to_clipboard(index=False)

In [186]:
city_benchmark=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (
                'Gurgaon Rural', 
                'NCR N sec 6354', 
                'Dashrathpuri_Del_East', 
                'NCR N sec 2262', 
                'NCR N Sharda University 2', 
                'NCR N Gharoli', 
                'NCR G Vijay Nagar 2', 
                'NCR N Venice Mall', 
                'Dwarka_Del', 
                'NCR N WaveCity Center'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
)








SELECT
        
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            cluster_name,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

                    
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            rr_net
        where 
            yyyymmdd is not null
           
        GROUP BY 1
        )
    ORDER BY 1
    
"""

city_benchmark_df=pd.read_sql(city_benchmark,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/3457448014.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  city_benchmark_df=pd.read_sql(city_benchmark,conn)


In [187]:
city_benchmark_df

,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,Dashrathpuri_Del_East,21963,49745,24594,49.44,68.05,110,136,343,186,18,0.59,21.09,18.30,10.26,293,10489,9103,5104
1,Dwarka_Del,6807,14768,8369,56.67,72.68,116,141,318,182,12,0.91,17.65,15.59,8.71,134,2607,2303,1287
2,Gurgaon Rural,10184,24206,11306,46.71,63.46,99,132,306,169,15,0.43,23.08,16.56,13.02,103,5587,4008,3152
3,NCR G Vijay Nagar 2,15108,31759,20789,65.46,82.94,82,132,276,179,11,0.68,12.49,17.05,3.88,216,3967,5414,1233
4,NCR N Gharoli,20586,43528,25924,59.56,79.88,101,130,321,199,14,0.54,14.03,19.86,5.53,237,6106,8644,2409
5,NCR N Sharda University 2,16499,47285,14783,31.26,44.85,82,109,264,126,10,1.30,37.65,13.19,16.19,616,17801,6238,7657
6,NCR N Venice Mall,11798,33623,13178,39.19,54.00,100,118,282,138,9,5.81,27.90,14.27,12.29,1952,9380,4799,4131
7,NCR N WaveCity Center,8797,17596,10641,60.47,80.77,98,127,306,192,11,0.83,13.40,19.93,4.99,146,2357,3507,878
8,Other_clusters,808280,2732646,1323003,48.41,66.87,107,132,311,176,14,1.65,19.92,18.08,11.54,45170,544351,494092,315265


In [191]:
city_benchmark_df.to_clipboard(index=False)

In [188]:
city_benchmark_s_c=f"""
with rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN pickup_cluster IN (
                'Gurgaon Rural', 
                'NCR N sec 6354', 
                'Dashrathpuri_Del_East', 
                'NCR N sec 2262', 
                'NCR N Sharda University 2', 
                'NCR N Gharoli', 
                'NCR G Vijay Nagar 2', 
                'NCR N Venice Mall', 
                'Dwarka_Del', 
                'NCR N WaveCity Center'
            ) THEN pickup_cluster
            ELSE 'Other_clusters'
        END AS cluster_name,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        case when SUBSTR(customer_id, -1) in ('1','3','5','7','9','e','a','d') then 'test' else 'control' end as test_group
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
        and customer_id in(
                select
                    user_id
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
            AND order_activity_source = 'appOpen'
            and current_city = 'Delhi'
            and platform='Android'
            and app_version>='8.30.0'
            
            
            )
)



SELECT
        time_period,
        cluster_name,
        customers,

        gross_count,
        net_count,
        
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",
        
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            cluster_name,
            COUNT(DISTINCT customer_id) customers,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,

                    
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            rr_net
        where 
            yyyymmdd is not null
           
        GROUP BY 1,2
        )
    ORDER BY 1,2
    
"""

city_benchmark_s_c_df=pd.read_sql(city_benchmark_s_c,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_31458/3988337630.py:129: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  city_benchmark_s_c_df=pd.read_sql(city_benchmark_s_c,conn)


In [169]:
city_benchmark_s_c_df

,time_period,cluster_name,customers,gross_count,net_count,g2n,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,afternoon,Gurgaon Rural,4683,8827,4261,48.27,65.82,97,129,342,176,16,0.24,22.82,17.36,11.12,21,2014,1532,982
1,afternoon,NCR N Gharoli,7834,11781,8528,72.39,92.36,93,108,302,230,10,0.31,5.35,19.43,1.98,37,630,2289,233
2,afternoon,NCR N Sharda University 2,9633,20524,7282,35.48,49.29,60,108,245,126,9,0.39,35.92,13.35,14.40,80,7372,2740,2955
3,afternoon,NCR N sec 22/62,5217,7748,6070,78.34,94.00,69,80,246,175,10,0.28,4.57,15.14,1.15,22,354,1173,89
4,afternoon,Other_clusters,382059,791101,452796,57.24,76.85,107,128,309,185,11,1.34,13.87,19.21,7.94,10582,109742,151978,62835
5,afternoon,Sec-10A,3659,6938,3176,45.78,62.90,50,136,332,174,14,0.25,23.28,16.86,13.58,17,1615,1170,942
6,afternoon,Sec-39,4393,7301,4578,62.70,81.56,109,130,275,183,11,0.19,12.14,18.59,6.11,14,886,1357,446
7,afternoon,Sec-55,3681,6365,3972,62.40,82.61,131,117,280,182,11,0.39,11.28,19.97,5.72,25,718,1271,364
8,afternoon,Sec-57,5709,9801,6027,61.49,80.98,89,126,277,180,11,0.45,12.34,19.21,6.23,44,1209,1883,611
9,afternoon,Sector 45-46,4835,7691,5278,68.63,87.23,54,118,278,190,11,0.27,9.34,18.41,3.16,21,718,1416,243


In [192]:
city_benchmark_s_c_df.to_clipboard(index=False)

In [18]:
click_bucket=f"""
with olf as (

    SELECT
        yyyymmdd,
        customer_id,
        order_id,
        order_requested_epoch,
        customer_cancelled_epoch,
        dropped_epoch,
        expired_epoch,
        rider_accepted_epoch
        
    FROM 
        orders.order_logs_fact
    WHERE
        yyyymmdd >= '20240911' AND yyyymmdd <= '20240916'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        
    ),




click as(
select
           yyyymmdd,
           profile_identity as user_id,
            epoch as click_epoch
        from
            clevertap.customer_notification_clicked_immutable 
        where yyyymmdd >= '20240911' AND yyyymmdd <= '20240916' 
        and event_props_wzrk_c2a='Button 2'
        and event_props_campaign_id = '1725970812' 
        
        -- group by 1,2
        
),


merge AS (
    
    SELECT 
        olf.*,
        click_epoch,
        (click_epoch / 1000 - order_requested_epoch / 1000) time_to_click
        --row_number() over(partition by olf.yyyymmdd,olf.customer_id,olf.order_id order by click_epoch) seq_no
    FROM 
        olf
    JOIN 
        click
        on olf.yyyymmdd = click.yyyymmdd
        and olf.customer_id = click.user_id
        and olf.order_requested_epoch < click.click_epoch
        and (olf.customer_cancelled_epoch > click.click_epoch or dropped_epoch > click.click_epoch or 
        expired_epoch > click.click_epoch or rider_accepted_epoch> click.click_epoch)
    )
    
    
   
    
    
    SELECT
        
        case 
        when time_to_click <= 15 then 'a. 15 Sec'
        when time_to_click between 16 and 30 then 'b. 16-30 Sec'
        when time_to_click between 31 and 60 then 'c. 30-60 Sec'
        when time_to_click between 61 and 120 then 'd. 1-2 min'
        when time_to_click between 121 and 240 then 'e. 3-4 min'
        when time_to_click > 240 then 'f. 4+ min' 
        else 'check' end as click_bucket,
        count(distinct customer_id) customers
    FROM 
        merge
    -- WHERE 
    --     --seq_no = 1
    GROUP BY 1
    ORDER BY 1

"""
click_bucket_df = pd.read_sql(click_bucket,conn)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62370/3278354793.py:84: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  click_bucket_df = pd.read_sql(click_bucket,conn)


In [20]:
click_bucket_df

,click_bucket,customers
0,a. 15 Sec,27299
1,b. 16-30 Sec,3613
2,c. 30-60 Sec,1438
3,d. 1-2 min,865
4,e. 3-4 min,659
5,f. 4+ min,519


In [24]:
click_bucket_df.to_clipboard(index=False)

In [22]:
32360-34393

-2033

In [23]:
27299-2033


25266

In [21]:
click_bucket_df.customers.sum()

34393

In [204]:
total_customers = click_bucket_df['customers'].sum()

# Calculate the percentage of customers for each click_bucket
click_bucket_df['customer_percentage'] = (click_bucket_df['customers'] / total_customers) * 100

# Round the percentage to 2 decimal places
click_bucket_df['customer_percentage'] = click_bucket_df['customer_percentage'].round(2)
